# Notebook 4b: Baseline 2, AdaBN (adaptive batch normalisation)

**What it is.** ResNet-50 has 53 *batch-normalisation* (BN) layers. Each one rescales the numbers flowing through the
network using an **average and a spread** that it memorised from the lab images during training. AdaBN
(Li et al., 2017) throws away those memorised statistics and **re-measures them on field images**. No weights are
trained, and the classifier is untouched.

**How the paper did it (Section 2.4).** "The batch normalization running statistics ... were replaced with statistics
computed from a forward pass over the Setting B calibration subset of PlantDoc images. All other model parameters,
including the classification head, remained unchanged." The calibration images are used **without labels**.

### In practice
For each of the 10 Setting B splits:
1. Start from the original trained network.
2. Run that split's ~200 calibration images through the network once, letting every BN layer measure the field images'
   averages and spreads.
3. Switch back to normal prediction mode and score **all** PlantDoc images (known and unknown).

Each split gives slightly different results, so we save one set of logits per split. This method has no Setting A.

### This notebook needs PyTorch and the PlantDoc images
But no training: it is quick (a few minutes) once the images are loaded.

### Plan
1. Settings
2. Load the saved outputs and the trained network
3. Load the PlantDoc images into memory
4. Check that we reproduce notebook 02's results
5. AdaBN on one split, looked at closely
6. All 10 splits, then save
7. Evaluate with notebook 3b

## Step 1: Settings

In [1]:
import copy
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torchvision import models, transforms

In [2]:
# ---- Edit these -------------------------------------------------------------
QUICK_TEST = False                 # True = only 2 splits, saved under a _QUICK name. Set False for the real run.

METHOD = "adabn"
SOURCE_METHOD = "source_only"     # whose trained network we start from
TRAIN_SEED = 42

OUT_DIR = Path("outputs")
MANIFEST = OUT_DIR / "master_manifest.csv"
PD_ROOT = Path(r"F:\mphil\pdoc")   # the same folder as in notebook 01
SEEDS = list(range(42, 52))       # the 10 Setting B splits

IMAGE_SIZE = 224
BN_BATCH = 50                     # calibration images per forward pass when re-measuring the BN statistics
PREDICT_BATCH = 128               # images per batch when predicting
# -----------------------------------------------------------------------------

seeds_to_run = SEEDS[:2] if QUICK_TEST else SEEDS
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device, "| splits to run:", seeds_to_run)
if QUICK_TEST:
    print("*** QUICK TEST: only 2 splits. Set QUICK_TEST = False for the real run. ***")

Using device: cuda | splits to run: [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]


## Step 2: Load the saved outputs and the trained network

From notebook 02 we need the PlantDoc **labels and image paths** (and the lab logits, which we pass through unchanged).
We also rebuild the ResNet-50 and load the trained weights, exactly as notebook 02b did.

In [3]:
source = np.load(OUT_DIR / f"outputs_{SOURCE_METHOD}_seed{TRAIN_SEED}.npz", allow_pickle=True)
classes = list(source["classes"])

known_paths = source["pd_known_paths"]
unknown_paths = source["pd_unknown_paths"]
field_labels = source["pd_known_labels"]
print(len(known_paths), "known and", len(unknown_paths), "unknown PlantDoc images")

# Which known image is calibration / test in each split (same linking as in notebook 3b)
manifest = pd.read_csv(MANIFEST, keep_default_na=False)
split_columns = [f"B_seed{s}" for s in SEEDS]
known_rows = manifest[(manifest["dataset"] == "pd") & (manifest["role"] == "pd_known")].set_index("path")
split_info = known_rows.loc[known_paths, split_columns].reset_index(drop=True)

2018 known and 466 unknown PlantDoc images


In [4]:
candidates = [p for p in sorted((OUT_DIR / "checkpoints").glob(f"{SOURCE_METHOD}_seed{TRAIN_SEED}*.pt"))
              if "DEBUG" not in p.name]
print("Checkpoint files found:", [p.name for p in candidates])
assert candidates, "No trained model found in outputs/checkpoints."

model = models.resnet50(weights=None)
model.fc = nn.Linear(model.fc.in_features, len(classes))
model.load_state_dict(torch.load(candidates[0], map_location=device))
model = model.to(device).eval()

original_state = copy.deepcopy(model.state_dict())      # a copy to restore before every split
n_bn = sum(isinstance(m, nn.BatchNorm2d) for m in model.modules())
print("Network loaded.", n_bn, "batch-normalisation layers will be re-measured.")

Checkpoint files found: ['source_only_seed42.pt']
Network loaded. 53 batch-normalisation layers will be re-measured.


## Step 3: Load the PlantDoc images into memory

We need the same images for all 10 splits, so we read and preprocess them **once** and keep them as one big array
(roughly 1.5 GB of memory). This is much faster than re-reading the files for every split.

In [5]:
eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


def load_images(relative_paths):
    tensors = []
    for i, p in enumerate(relative_paths):
        tensors.append(eval_transform(Image.open(PD_ROOT / p).convert("RGB")))
        if (i + 1) % 500 == 0:
            print(f"  {i + 1} of {len(relative_paths)} images loaded")
    return torch.stack(tensors)


start = time.time()
known_images = load_images(known_paths)
unknown_images = load_images(unknown_paths)
print(f"known: {tuple(known_images.shape)}, unknown: {tuple(unknown_images.shape)} "
      f"({(time.time() - start):.0f} s)")

  500 of 2018 images loaded
  1000 of 2018 images loaded
  1500 of 2018 images loaded
  2000 of 2018 images loaded
known: (2018, 3, 224, 224), unknown: (466, 3, 224, 224) (60 s)


## Step 4: Check that we reproduce notebook 02

Before changing anything, the network should give (almost) the same scores as the ones saved in notebook 02. If it
doesn't, something is off (wrong checkpoint, wrong image order, wrong preprocessing), and everything after would be
meaningless. This is the most important check in the notebook.

In [6]:
@torch.no_grad()
def predict(images):
    model.eval()                                  # normal prediction mode: BN uses its stored statistics
    out = []
    for s in range(0, len(images), PREDICT_BATCH):
        out.append(model(images[s:s + PREDICT_BATCH].to(device)).cpu().numpy())
    return np.concatenate(out)


check = predict(known_images[:256])
saved = source["pd_known_logits"][:256]
agree = float((check.argmax(1) == saved.argmax(1)).mean())
print(f"Top class agrees with notebook 02 on {agree * 100:.1f}% of 256 images")
print(f"Largest difference between logits: {np.abs(check - saved).max():.4f}")
assert agree > 0.98, "Does not match notebook 02: check the checkpoint, PD_ROOT and image order"

Top class agrees with notebook 02 on 100.0% of 256 images
Largest difference between logits: 0.0000


## Step 5: AdaBN on one split, looked at closely

`adapt_bn` does three things:
1. **Reset** every BN layer's stored average and spread.
2. Set `momentum = None`, which tells PyTorch to simply **average** over everything it sees (instead of a slow-moving
   average that would favour the last images).
3. Put the network in *training mode* and run the calibration images through it. In training mode each BN layer
   measures the statistics of the batch it receives and updates its stored values. `torch.no_grad()` means nothing is
   *learned*, only measured.

Then we switch back to prediction mode.

In [7]:
def adapt_bn(network, calibration_images):
    for m in network.modules():
        if isinstance(m, nn.BatchNorm2d):
            m.reset_running_stats()               # forget the lab statistics
            m.momentum = None                     # None = plain average over all batches seen

    network.train()                               # BN layers measure the statistics of what they receive
    with torch.no_grad():                         # ... but nothing is learned
        for s in range(0, len(calibration_images), BN_BATCH):
            network(calibration_images[s:s + BN_BATCH].to(device))
    network.eval()                                # back to prediction mode, now using the field statistics

In [8]:
seed = 42
is_cal = (split_info[f"B_seed{seed}"] == "cal").to_numpy()
is_test = ~is_cal

model.load_state_dict(original_state)                       # start from the original lab statistics
lab_mean = model.bn1.running_mean.clone()                   # peek at the very first BN layer

adapt_bn(model, known_images[torch.from_numpy(is_cal)])     # re-measure on this split's calibration images only
print(f"Re-measured BN statistics from {is_cal.sum()} calibration images (no labels used)")
print("First BN layer: average change in its stored mean:",
      round(float((model.bn1.running_mean - lab_mean).abs().mean()), 4))

C:\Users\MIKE\AppData\Local\Temp\ipykernel_156\1581562354.py:8: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  adapt_bn(model, known_images[torch.from_numpy(is_cal)])     # re-measure on this split's calibration images only


Re-measured BN statistics from 202 calibration images (no labels used)
First BN layer: average change in its stored mean: 0.2238


In [9]:
adapted = predict(known_images)
original = source["pd_known_logits"]

def accuracy(logits, mask):
    return float((logits[mask].argmax(axis=1) == field_labels[mask]).mean())

print(f"Accuracy on the {is_test.sum()} test images (labels used only for scoring):")
print(f"  original network: {accuracy(original, is_test):.4f}")
print(f"  with AdaBN:       {accuracy(adapted, is_test):.4f}")

Accuracy on the 1816 test images (labels used only for scoring):
  original network: 0.2390
  with AdaBN:       0.3243


**Questions to think about:**
* The paper reports about 0.321 for the original model and 0.343 for AdaBN. Is your change in that direction and of that size?
* Which parts of the domain gap can re-measuring averages and spreads fix? Which can't it?
* We re-measured on only ~200 images. What could go wrong with such a small sample?

## Step 6: All splits, then save

For each split: restore the original network, re-measure the BN statistics on that split's calibration images, score
**all** known and unknown PlantDoc images, and keep the logits. The lab logits are copied unchanged from notebook 02.

In [10]:
save_dict = {"classes": np.array(classes)}
for key in ["epochs_run", "debug_mode"]:
    if key in source.files:
        save_dict[key] = source[key]
for g in ["pv_val", "pv_test"]:
    save_dict[f"{g}_logits"] = source[f"{g}_logits"]
for g in ["pv_val", "pv_test", "pd_known", "pd_unknown"]:
    save_dict[f"{g}_labels"] = source[f"{g}_labels"]
    save_dict[f"{g}_paths"] = source[f"{g}_paths"]

for seed in seeds_to_run:
    start = time.time()
    is_cal = (split_info[f"B_seed{seed}"] == "cal").to_numpy()

    model.load_state_dict(original_state)                    # always start from the original network
    adapt_bn(model, known_images[torch.from_numpy(is_cal)])

    save_dict[f"pd_known_logits_B_seed{seed}"] = predict(known_images).astype(np.float32)
    save_dict[f"pd_unknown_logits_B_seed{seed}"] = predict(unknown_images).astype(np.float32)
    print(f"split {seed}: {time.time() - start:.0f} s")

model.load_state_dict(original_state)                        # leave the network as we found it

suffix = "_QUICK" if QUICK_TEST else ""
output_file = OUT_DIR / f"outputs_{METHOD}_seed{TRAIN_SEED}{suffix}.npz"
np.savez_compressed(output_file, **save_dict)
print("Saved", output_file)

split 42: 5 s
split 43: 5 s
split 44: 5 s
split 45: 5 s
split 46: 5 s
split 47: 5 s
split 48: 5 s
split 49: 5 s
split 50: 5 s
split 51: 5 s
Saved outputs\outputs_adabn_seed42.npz


## Step 7: Evaluate with notebook 3b

Run this notebook once with `QUICK_TEST = True` to check that it works, then set it to `False`, restart the kernel and
run it again. The real file is named `outputs_adabn_seed0.npz` (no `_QUICK`). Then in notebook 3b add `"adabn"` to
`METHODS`.

**The paper's reference values for AdaBN** (Table 10): accuracy 0.343, calibrated ECE 0.288, calibrated Brier 0.911,
unknown AUROC 0.611, AUPR 0.260, Risk@80 0.623, Risk@50 0.553.

### Next
**Notebook 4c: DANN**, the adversarial baseline that the paper found to make things *worse*.